# Milestone 3 — M1 Parquet Export, Sharded Candidates, Auto-Labeling & Predictions (Member 1)

Run on a **Google Colab CPU** runtime. Each step is an independent cell with bounded memory.

1. **Setup & verified restore:** copies only filtered M2 `bibtex.parquet` shards and M1 `metadata.json` + `references.json` (SHA-256 verified against `INDEX.json`).
2. **Export M1 Parquet tables:** writes `papers` and `references` in sharded zstd Parquet conforming to the M1/M3 schema (`data/papers/*.parquet`, `data/references/*.parquet`).
3. **Sharded candidate matrix:** builds `(arxiv_id, citation_key, ref_arxiv_id)` shards without loading the full BibTeX table into RAM.
4. **Rule-based auto-labeler:** labels at least 10% of non-manual publications (default 2,000 papers, stratified across all 4 ranges) into `labels/auto/*.csv`, and evaluates precision/recall against `labels/manual/*.csv`.
5. **Predictions table export:** generates schema-validated `predictions` Parquet shards once `partitions.csv` and the trained ranker are ready.

In [ ]:
M1_DRIVE_DIR = "/content/drive/MyDrive/arxiv-nhom1"
M2_DRIVE_DIR = "/content/drive/MyDrive/arxiv-nhom1/m2"
M2_SOURCE_RUN_ID = "m2-full-01"
M2_DERIVED_ID = "m2-full-no-empty-01"
M3_DRIVE_DIR = "/content/drive/MyDrive/arxiv-nhom1/m3"
CODE_REF = "main"
REPO = "https://github.com/hcmus-arxiv-nhom1/CodeRepo.git"
LOCAL = "/content/m3-thien-local"

## 1. Mount Drive, checkout code, verify M2 derived identity & restore inputs

In [ ]:
import hashlib
import json
import os
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
repo_dir = Path("/content/CodeRepo-m3-thien")
if not (repo_dir / ".git").is_dir():
    subprocess.run(["git", "clone", REPO, str(repo_dir)], check=True)
subprocess.run(["git", "-C", str(repo_dir), "fetch", "origin", CODE_REF], check=True)
subprocess.run(["git", "-C", str(repo_dir), "checkout", "--detach", "FETCH_HEAD"], check=True)
os.chdir(repo_dir)
if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))
print("code revision:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "pyarrow", "pandas", "tqdm"],
    check=True,
)

M2_SOURCE_RUN = Path(M2_DRIVE_DIR) / M2_SOURCE_RUN_ID
M2_RUN = Path(M2_DRIVE_DIR) / "derived" / M2_DERIVED_ID
M3 = Path(M3_DRIVE_DIR)
LABELS = M3 / "labels"
M2_OUT = Path(LOCAL) / M2_DERIVED_ID / "out"
CORPUS = Path(LOCAL) / "corpus"
M3.mkdir(parents=True, exist_ok=True)

In [ ]:
from tqdm.auto import tqdm

from src.m3.restore_inputs import restore_bibtex, restore_m1_json

identity = json.loads((M2_RUN / "identity.json").read_text(encoding="utf-8"))
manifest = (M2_SOURCE_RUN / "M2_MANIFEST.json").read_bytes()
if (
    identity["run_id"] != M2_SOURCE_RUN_ID
    or identity["manifest_sha256"] != hashlib.sha256(manifest).hexdigest()
):
    raise ValueError("Derived M2 output does not match M2_MANIFEST.json")


def progress(bar):
    def update(done, total):
        bar.total = total
        bar.update(done - bar.n)

    return update


with tqdm(desc="bibtex shards", unit="shard") as bar:
    print(restore_bibtex(M2_RUN, M2_OUT, on_progress=progress(bar)))
with tqdm(desc="M1 metadata + references", unit="file") as bar:
    print(restore_m1_json(M2_SOURCE_RUN, M1_DRIVE_DIR, CORPUS, on_progress=progress(bar)))

## 2. Export M1 `papers` & `references` Parquet tables (all 4 ranges)

In [ ]:
from src.m3 import export_m1_tables

M1_PARQUET_ALL = M3 / "m1_tables_all"
M1_PARQUET_SPLIT = M3 / "m1_tables_by_range"

# Combined sharded Parquet (fast input for M3 candidate matrix & auto-labeler)
export_m1_tables.main(
    [
        "--corpus-dir",
        str(CORPUS),
        "--out-dir",
        str(M1_PARQUET_ALL),
        "--assigned-only",
        "--shard-papers",
        "500",
    ]
)

# Per-member sharded Parquet (<owner>/data/{papers,references}/*.parquet) for HF packaging
export_m1_tables.main(
    [
        "--corpus-dir",
        str(CORPUS),
        "--out-dir",
        str(M1_PARQUET_SPLIT),
        "--split-by-range",
        "--shard-papers",
        "500",
    ]
)

## 3. Build sharded candidate matrices (bounded RAM)

In [ ]:
from src.m3 import candidates

REFS_PARQUET = M1_PARQUET_ALL / "data" / "references"
CANDIDATES_DIR = M3 / "candidates"

candidates.main(
    [
        "--m2-dir",
        str(M2_OUT),
        "--refs-dir",
        str(REFS_PARQUET),
        "--out-dir",
        str(CANDIDATES_DIR),
    ]
)

## 4. Run rule-based auto-labeler (>=10% of non-manual papers) & evaluate on manual labels

Make sure `selection.csv` (the 20 manual papers) is already in `M3_DRIVE_DIR` before running `auto_label run` so manual papers are strictly excluded.

In [ ]:
from src.m3 import auto_label

SELECTION = M3 / "selection.csv"
auto_args = [
    "run",
    "--m2-dir",
    str(M2_OUT),
    "--refs-dir",
    str(REFS_PARQUET),
    "--labels-dir",
    str(LABELS),
    "--total-corpus-papers",
    "20000",
    "--ratio",
    "0.10",
    "--min-papers",
    "2000",
    "--seed",
    "20261117",
]
if SELECTION.exists():
    auto_args += ["--selection-csv", str(SELECTION)]
auto_label.main(auto_args)

In [ ]:
# Run once manual label CSVs exist under M3_DRIVE_DIR/labels/manual/
if any((LABELS / "manual").glob("*.csv")):
    auto_label.main(
        [
            "evaluate",
            "--m2-dir",
            str(M2_OUT),
            "--refs-dir",
            str(REFS_PARQUET),
            "--labels-dir",
            str(LABELS),
            "--out-dir",
            str(M3 / "auto_eval"),
        ]
    )
else:
    print("No manual CSVs found under", LABELS / "manual")

## 5. Generate `predictions` Parquet table (once `partitions.csv` and trained ranker exist)

Replace `RANKER = "baseline"` with the trained model's `module:function` once ready.

In [ ]:
from src.m3 import predictions

PARTITIONS = M3 / "partitions.csv"
RANKER = "baseline"  # e.g. "src.m3.model:rank_paper"
if PARTITIONS.exists():
    predictions.main(
        [
            "--m2-dir",
            str(M2_OUT),
            "--refs-dir",
            str(REFS_PARQUET),
            "--labels-dir",
            str(LABELS),
            "--partitions",
            str(PARTITIONS),
            "--out-dir",
            str(M3 / "predictions_by_range"),
            "--ranker",
            RANKER,
            "--split-by-range",
        ]
    )
else:
    print("Waiting for partitions.csv at", PARTITIONS)